In [ ]:
from repo_paths import DATA_PATH

import tabula
import pandas as pd

pdf_tables = tabula.read_pdf('https://openknowledge.fao.org/server/api/core/bitstreams/836a65eb-1f12-4286-b62e-77ba4beedf16/content', pages='all', multiple_tables=True, lattice=True)
# choose dfs from array that have 7 columns, they're the relevant tables
food_group_dfs = [df for df in pdf_tables if df.shape[1] == 7]

#concatenate all dfs
food_groups = pd.concat(food_group_dfs, ignore_index=True)
food_groups.columns = [
    'group_code',
    'group_name_long',
    'group_name_short',
    'subgroup_code',
    'subgroup_name_long',
    'subgroup_name_short',
    'subgroup_description'
]
food_groups.replace(to_replace=[r'\r', r'\n'], value=' ', regex=True, inplace=True)
food_groups['group_code'] = food_groups['group_code'].astype('Int64')

In [ ]:
#select all rows where first five cols are na, these result from the description flowing over to the next page
page_break_mask = food_groups.iloc[:, :5].isna().all(axis=1)

In [ ]:
#function to move contents to the right where cell merging in the pdf table has caused it to be read in the wrong place
def shift_columns_right(df, mask, shift_distance, num_cols_to_shift):
    cols = df.columns
    
    for i in range(len(cols)-1, shift_distance-1, -1):
        df.loc[mask, cols[i]] = df.loc[mask, cols[i-shift_distance]]
    
    df.loc[mask, cols[:num_cols_to_shift-1]] = pd.NA


In [ ]:
#some rows only shifted by 1
shifted1_mask = food_groups['group_name_short'].fillna('').str.isnumeric()
shift_columns_right(food_groups, shifted1_mask, 1, 4)

In [ ]:
#in most cases the subgroup code got read into the first col, and we shift by 3
shifted3_mask = food_groups['group_code'] > 100
shift_columns_right(food_groups, shifted3_mask, 3, 4)

In [ ]:
# For each row where page_break_mask is True, concatenate the last col to the previous row and delete

for idx in food_groups.index[page_break_mask]:
    food_groups.at[idx-1, 'subgroup_description'] = str(food_groups.at[idx-1, 'subgroup_description']) + ' ' + str(food_groups.at[idx, 'subgroup_description'])

food_groups.drop(food_groups.index[page_break_mask], inplace=True)


In [ ]:
food_groups['subgroup_code'] = pd.to_numeric(food_groups['subgroup_code'], errors='coerce').astype('Int64')

left_3_cols = ['group_code', 'group_name_long', 'group_name_short']

food_groups.loc[:, left_3_cols] = food_groups.loc[:, left_3_cols].ffill()

#export to csv
food_groups.to_csv(DATA_PATH+'/fao_groups.csv', index=False)